# ML Foundations I: learning from data, fitting a line, improving and judging a model

The three pages of chapter 4, run in code. Open **one** notebook, run the setup cell, then any section; each matches a page, in order.

**No API key and nothing to download.** Everything is numpy, pandas and matplotlib on the firm's illustrative monthly returns (made up, not market data). Random numbers are never used, so you will get the same output as the textbook.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt

# --- the same illustrative data and helpers the textbook's in-page cells use (ml_basics.py) ---
MONTHS = ["Oct 24", "Nov 24", "Dec 24", "Jan 25", "Feb 25", "Mar 25", "Apr 25", "May 25", "Jun 25", "Jul 25", "Aug 25", "Sep 25",
          "Oct 25", "Nov 25", "Dec 25", "Jan 26", "Feb 26", "Mar 26", "Apr 26", "May 26", "Jun 26", "Jul 26", "Aug 26", "Sep 26"]

# Monthly returns, as fractions (0.0086 = +0.86%).
MARKET = [0.0086, -0.0166, 0.0233, 0.0585, 0.0706, 0.032, 0.01, -0.0021, -0.0093, -0.032, -0.0531, -0.0618, 0.0478, 0.0058, 0.0462, -0.0464, 0.018, 0.0074, 0.0339, -0.0012, 0.0442, 0.0462, 0.0594, -0.0718]
SIZE = [-0.0237, 0.0292, 0.0526, 0.0111, -0.0058, 0.0251, 0.0053, 0.0052, 0.028, 0.0023, -0.0331, -0.0128, 0.0135, -0.0159, 0.0087, -0.01, 0.0121, -0.0093, 0.0193, -0.0063, -0.0207, 0.0116, 0.0084, 0.0039]
VALUE = [-0.036, -0.009, 0.0265, 0.0091, -0.0278, 0.0338, 0.0129, 0.0013, -0.0093, -0.0012, 0.0146, -0.0564, -0.0069, -0.0298, 0.0129, 0.0048, 0.0027, 0.0123, 0.0442, 0.0491, -0.0294, 0.0465, 0.0049, -0.0411]
MOMENTUM = [0.0436, -0.0202, -0.0155, 0.0316, -0.0019, 0.0089, 0.0414, 0.0146, -0.0756, -0.0074, -0.0363, -0.0341, -0.0352, 0.0098, 0.0072, -0.0032, 0.0356, 0.0078, -0.0043, 0.0346, 0.0208, 0.0111, 0.0375, 0.0278]
RISK_FREE = [0.0035] * 24   # about 4.3% a year, paid monthly

RETURNS = {
    "DE": [0.0002, -0.0156, 0.0303, 0.0429, 0.143, 0.0623, -0.0545, 0.0035, -0.0284, -0.0655, -0.0669, -0.0805, 0.0887, -0.0112, 0.0538, -0.0353, -0.0482, -0.0356, 0.0137, 0.0464, 0.0277, 0.0372, 0.0427, -0.1365],
    "CAT": [0.0086, -0.0313, 0.0492, 0.0959, 0.1035, 0.0601, -0.0025, 0.0047, -0.0242, -0.0444, -0.0278, -0.1255, 0.0728, 0.0096, 0.063, -0.0777, 0.0231, 0.0034, 0.0224, -0.0186, -0.004, 0.0588, 0.1594, -0.1386],
    "MKT": MARKET,
}
# Month-end price before the first return, so that the last price is today's price in tools.PRICES (DE 512.40, CAT 398.15).

# One row per month. Features are the benchmark returns; the label is Deere's return.
FEATURES = ["market", "size", "value", "momentum"]


def table(label="DE"):
    """(X, y): X is a list of rows [market, size, value, momentum], y the label for that month, all in percent."""
    X = [[100 * m, 100 * s, 100 * v, 100 * mo] for m, s, v, mo in zip(MARKET, SIZE, VALUE, MOMENTUM)]
    y = [100 * r for r in RETURNS[label]]
    return X, y


def column(X, j):
    """One feature as a list: the j-th number of every row."""
    return [row[j] for row in X]


def predict(w, b, row):
    """The model: a weighted sum of the features plus a starting point (a dot product)."""
    return sum(wi * xi for wi, xi in zip(w, row)) + b


def mse(y, pred):
    """Mean squared error: square each miss, then average. Big misses count far more."""
    return sum((a - p) ** 2 for a, p in zip(y, pred)) / len(y)


def mae(y, pred):
    """Mean absolute error: average size of a miss, no squaring."""
    return sum(abs(a - p) for a, p in zip(y, pred)) / len(y)


def fit_line(x, y):
    """Least squares for one feature: the slope and intercept with the smallest mean squared error (closed form)."""
    mx, my = sum(x) / len(x), sum(y) / len(y)
    slope = sum((a - mx) * (c - my) for a, c in zip(x, y)) / sum((a - mx) ** 2 for a in x)
    return slope, my - slope * mx


def gradient_step(X, y, w, b, lr):
    """One step of gradient descent: nudge every parameter a little in the direction that lowers the MSE."""
    n = len(y)
    err = [predict(w, b, row) - t for row, t in zip(X, y)]
    gw = [2 / n * sum(e * row[j] for e, row in zip(err, X)) for j in range(len(w))]
    gb = 2 / n * sum(err)
    return [wi - lr * g for wi, g in zip(w, gw)], b - lr * gb


def descend(X, y, lr=0.01, steps=50, w=None, b=0.0):
    """Run gradient descent from a starting guess; returns the history [(step, loss, w, b), ...]."""
    w = list(w) if w is not None else [0.0] * len(X[0])
    hist = [(0, mse(y, [predict(w, b, r) for r in X]), list(w), b)]
    for s in range(1, steps + 1):
        w, b = gradient_step(X, y, w, b, lr)
        hist.append((s, mse(y, [predict(w, b, r) for r in X]), list(w), b))
    return hist


def split_by_date(X, y, test_months=6):
    """Earlier months to learn from, the last few held back. Never shuffled: a model is used on months after the ones it studied."""
    k = len(y) - test_months
    return X[:k], y[:k], X[k:], y[k:]


## 1 · Learning from data (page 4.1)

In [ ]:
X, y = table()
print("features:", FEATURES)
print("rows:", len(X), " columns:", len(X[0]))
df = pd.DataFrame(X, columns=FEATURES, index=MONTHS)
df["Deere same month"] = y
print(df.round(2).head(6))

In [ ]:
# the next-month version: this month's features, NEXT month's Deere return
Xn, yn = X[:-1], y[1:]
print("vector :", [round(v, 2) for v in Xn[3]])
print("label  :", round(yn[3], 2))

w, b = [1.0, 0.0, 0.0, 0.0], 0.0       # a guess: one-for-one with the market
for k in (0, 3, 11):
    print("month", k, "predicted", round(predict(w, b, X[k]), 2), "actual", round(y[k], 2))

**Try it.** Use `table("CAT")`. Which of the four columns moves most like the labels? Check with `np.corrcoef`.

## 2 · Fitting a line (page 4.2)

In [ ]:
x = column(X, 0)
slope, intercept = fit_line(x, y)
pred = [slope * a + intercept for a in x]
print(f"best line: Deere = {slope:.2f} x market + ({intercept:.2f})")
print("MSE:", round(mse(y, pred), 1), "  MAE:", round(mae(y, pred), 1))
print("guess-the-average MSE:", round(mse(y, [sum(y) / len(y)] * len(y)), 1))

plt.scatter(x, y, color="#13294b")
xs = np.array([min(x), max(x)])
plt.plot(xs, slope * xs + intercept, color="#ff5f05")
plt.xlabel("market, % (same month)"); plt.ylabel("Deere, %"); plt.show()

In [ ]:
# four features, found by walking (section 3 explains how)
h = descend(X, y, lr=0.002, steps=3000)
w, b = h[-1][2], h[-1][3]
print("weights:", [round(v, 2) for v in w], " bias:", round(b, 2))
print("four-feature MSE:", round(mse(y, [predict(w, b, r) for r in X]), 1), "(one feature: 9.7)")

**Try it.** Refit for `table("CAT")`. Then drop the last month (the -7.2 market, -13.7 Deere) and refit. How far does the slope move?

## 3 · Improving and judging a model (page 4.3)

In [ ]:
x1 = [[a] for a in column(X, 0)]
for lr in (0.001, 0.02, 0.07):
    h = descend(x1, y, lr=lr, steps=30)
    print(f"step size {lr}: loss at 0, 1, 5, 30 steps =", [round(h[k][1], 1) for k in (0, 1, 5, 30)])

h = descend(x1, y, lr=0.02, steps=60)
plt.plot([r[1] for r in h], color="#13294b"); plt.xlabel("step"); plt.ylabel("loss"); plt.show()

In [ ]:
Xn, yn = X[:-1], y[1:]

def score(cols, test_months=6):
    rows = [[r[j] for j in cols] for r in Xn]
    Xa, ya, Xb, yb = split_by_date(rows, yn, test_months)
    h = descend(Xa, ya, lr=0.002, steps=4000)
    w, b = h[-1][2], h[-1][3]
    return mse(ya, [predict(w, b, r) for r in Xa]), mse(yb, [predict(w, b, r) for r in Xb]), mse(yb, [sum(ya) / len(ya)] * len(yb))

for name, cols in (("1 feature", [0]), ("4 features", [0, 1, 2, 3])):
    tr, te, base = score(cols)
    print(f"{name}: learned {tr:5.1f}   held back {te:5.1f}   just-the-average {base:5.1f}")

**Try it.** Change `test_months` to 3, then 9. Does the verdict on the four-feature model change? What does that say about trusting six months of test data?